In [1]:
import uuid
import tempfile
from typing import Any, Dict, List
import ray, os, time
import collections
from ray import ObjectRef
from ray.actor import ActorHandle

/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-26 11:27:59,886	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
ray.init()  # 单机默认：多进程并行

2026-04-26 11:28:03,817	INFO worker.py:2012 -- Started a local Ray instance.
/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.0
Ray version:,2.51.1


### basics

> Agentic Infra 的复杂性和魅力

- `@ray.remote` 注释函数：ray task，`@ray.remote` 注释类：ray actor
    - @ray.remote class 为 Actor,
        - ray actor，在独立的进程执行；
        - 有状态远程对象、长活服务
    - @ray.remote function 为 task
        - 大规模并行、跨节点、DAG 编排
- actor 的特点是：它有状态（类实例化后的成员变量）。这个状态活在 Ray worker 进程里，不在当前 Python 进程里。
    - 在当前 Python 进程里拿到的不是对象本身，而是一个“远程对象句柄”。

In [27]:
@ray.remote
class A:
    def __init__(self, init_value: int = 0):
        self.value = init_value

    def some_method(self, delta: int) -> int:
        self.value += delta
        return self.value

    def get_value(self) -> int:
        return self.value

In [28]:
# ActorHandle，相当于“远程对象的遥控器”。
a = A.remote(10)
a, type(a), isinstance(a, ActorHandle), isinstance(a, ObjectRef)

(Actor(A, cc257271e8e3645efd622a2e01000000),
 ray.actor.ActorHandle,
 True,
 False)

In [29]:
# ObjectRef[int]，相当于“这次远程调用结果的取货单”。
r = a.some_method.remote(5)

In [30]:
type(r)

ray.ObjectRef

In [31]:
value = ray.get(r)
value

15

- 传统：y = f(x) → 立即在本进程执行并返回结果。
- Ray：ref = f.remote(x) → **异步提交任务**，立即返回一个 ObjectRef（未来值/句柄）；随后用 ray.get(ref) 才会把结果取回本地。
    - ObjectRef 只是引用，不拷贝大对象本体。把 ObjectRef 当参数传给另一个任务，Ray 会自动建立依赖边，形成 DAG。
        - ObjectRef 约等于 Future
        - 依赖即调度：把 ObjectRef 作为参数传给下游任务，Ray 会自动识别拓扑顺序，只有当依赖就绪才执行后继节点；没有依赖的节点可最大化并行。

In [5]:
@ray.remote
def pid_job(i):
    time.sleep(0.2)
    return i, os.getpid()

refs = [pid_job.remote(i) for i in range(8)]
results = ray.get(refs)

# 看看有多少不同的进程在执行
pids = {pid for _, pid in results}
print("unique worker pids:", pids, "count:", len(pids))

unique worker pids: {1609028, 1609034, 1609009, 1609010, 1609016, 1609017, 1609019, 1609023} count: 8


- 基于多进程（Multi-processing）的任务级并行计算（Task-level Parallelism）。
    - 绕过 GIL 的物理隔离 (Process Allocation & Isolation)
    - 异步提交与非阻塞执行 (Asynchronous Futures)
        - 在执行 `refs = [pid_job.remote(i) for i in range(8)]` 的列表推导式时，代码实际上是在几毫秒内瞬间完成遍历的。
        - @ray.remote 将原本同步阻塞的函数转换为了异步（Asynchronous）的 Future 模式。remote() 方法的调用会立即返回一个对象引用（ObjectRef），而真正的睡眠/运算逻辑在后台进程中悄然进行。直到执行到 ray.get(refs) 时，主程序才会在此处阻塞，等待并收集所有工作进程的最终汇报。
    - Task（无状态任务）的动态路由分发
        - 在 Ray 的架构语义中，此处的 pid_job 函数被称为一个 Task（无状态任务）。

#### `ray.remote` 对比 python 多线程、多进程 or 协程

- 并发与并行
    - 并发（Concurrency）：强调“逻辑上的同时处理”。
        - 并发的本质是对 I/O 空闲等待时间的巧妙交叠利用。
    - 并行（Parallelism）：强调“物理上的同时执行”。
- Ray 并非仅仅是在单机上对多进程的语法糖封装（跨节点，多机），它是一个无中心化、具备分布式内存支持的任务/状态调度框架。装饰器 @ray.remote 的作用是实现“函数/类”到“分布式任务（Task）/分布式对象（Actor）”的语义升维。（分布式版的“进程池 + Future + Actor 模型 + 对象存储”。）
    - 多线程：依赖操作系统进行上下文切换的抢占式（Preemptive）调度。
    - 多进程：通过在操作系统层面 fork 出拥有独立内存空间的全新 Python 解释器进程，彻底绕开 GIL 的桎梏，实现物理级别的真正并行。
    - 协程（单线程内的“时间切片大师”）：协程是一种非抢占式（Non-preemptive）的用户态并发机制。它依赖于事件循环（Event Loop），在执行到达 I/O 阻塞点时，任务主动让出（Yield）控制权。
- `ray.remote` vs. RPC (remote procedure call)
    - “RPC 风格的接口 + 分布式任务/Actor 运行时”

### ray task

In [6]:
@ray.remote
def preprocess(batch_id: int, payload: List[int]) -> Dict[str, Any]:
    """
    无状态的纯计算任务，适合并行。
    返回一些统计数据，模拟重活（sleep）。
    """
    time.sleep(0.2 + 0.1 * (batch_id % 3))  # 模拟不同耗时
    s = sum(payload)
    mx = max(payload) if payload else None
    return {"batch_id": batch_id, "sum": s, "max": mx, "size": len(payload)}

In [7]:
batches = {
    0: list(range(10)),
    1: [5, 6, 7],
    2: [42],
    3: list(range(100, 110)),
    4: []
}
refs = [preprocess.remote(bid, arr) for bid, arr in batches.items()]
remaining = set(refs)
results = []
while remaining:
    done, pending = ray.wait(list(remaining), num_returns=1, timeout=None)
    remaining -= set(done)
    out = ray.get(done[0])
    print(f"[Task Done] batch_id={out['batch_id']}, sum={out['sum']}, size={out['size']}")
    results.append(out)

# 你也可以一次性 ray.get(refs) 获得无序列表，这里刻意用 wait 展示“完成即消费”
print("[Task Summary]", results)

[Task Done] batch_id=0, sum=45, size=10
[Task Done] batch_id=3, sum=1045, size=10
[Task Done] batch_id=4, sum=0, size=0
[Task Done] batch_id=1, sum=18, size=3
[Task Done] batch_id=2, sum=42, size=1
[Task Summary] [{'batch_id': 0, 'sum': 45, 'max': 9, 'size': 10}, {'batch_id': 3, 'sum': 1045, 'max': 109, 'size': 10}, {'batch_id': 4, 'sum': 0, 'max': None, 'size': 0}, {'batch_id': 1, 'sum': 18, 'max': 7, 'size': 3}, {'batch_id': 2, 'sum': 42, 'max': 42, 'size': 1}]


In [34]:
@ray.remote
def slow_square(x):
    time.sleep(1)
    return (x * x, os.getpid())   # 返回结果和执行该任务的进程 PID

@ray.remote
def add(a, b):
    # 这里 a、b 可以是普通值，也可以是 ObjectRef（依赖会被自动处理）
    return a + b

In [35]:
refs = [slow_square.remote(i) for i in range(4)]
print("立即返回的 ObjectRefs:", refs)

立即返回的 ObjectRefs: [ObjectRef(f2de7ac0316578aaffffffffffffffffffffffff0100000001000000), ObjectRef(878b2330f6be64f9ffffffffffffffffffffffff0100000001000000), ObjectRef(38e143cc5dfd1165ffffffffffffffffffffffff0100000001000000), ObjectRef(877e0687cd603f1affffffffffffffffffffffff0100000001000000)]


In [10]:
# 分两步取结果：先把平方值取出，再相加
vals = ray.get(refs)  # 等待全部完成
squares = [v for v, pid in vals]
print("各次平方结果与执行PID:", vals)

各次平方结果与执行PID: [(0, 1609010), (1, 1609016), (4, 1609034), (9, 1609017)]


In [11]:
# 构造依赖：把前两个任务的结果作为 add 的输入
sum_ref = add.remote(squares[2], squares[3])
print("两数相加的结果:", ray.get(sum_ref))

ray.shutdown()

两数相加的结果: 13


In [36]:
ray.get(add.remote(refs[2], refs[3]))

(4, 1641251, 9, 1641269)

### Map-Reduce (DAG)

In [12]:
@ray.remote
def map_count(text_chunk: str):
    cnt = collections.Counter()
    for w in text_chunk.lower().split():
        cnt[w] += 1
    return cnt

@ray.remote
def reduce_merge(a: collections.Counter, b: collections.Counter):
    a.update(b)
    return a

texts = [
    "Ray makes distributed computing simple simple",
    "Ray tasks are easy to scale",
    "Remote functions return ObjectRefs",
    "ObjectRefs build a DAG of dependencies"
]

# Map 阶段（并行）
parts = [map_count.remote(t) for t in texts]

# Reduce 阶段（树形合并，依赖于前序结果）
# [(0, 1), (2, 3)] => [((0, 1), (2, 3))]
while len(parts) > 1:
    nxt = []
    for i in range(0, len(parts), 2):
        if i+1 < len(parts):
            nxt.append(reduce_merge.remote(parts[i], parts[i+1]))
        else:
            nxt.append(parts[i])
    parts = nxt

final_counter = ray.get(parts[0])
print(final_counter.most_common(5))

ray.shutdown()

2026-04-26 11:28:13,453	INFO worker.py:2012 -- Started a local Ray instance.


[('ray', 2), ('simple', 2), ('objectrefs', 2), ('makes', 1), ('distributed', 1)]


### ray actor

#### stateful

In [13]:
@ray.remote
class StatefulCounter:
    def __init__(self):
        # 内部状态：独立账本
        self.count = 0
        # 记录该 Actor 被实例化时被分配的物理进程 ID
        self.pid = os.getpid()

    def increment_and_get(self):
        self.count += 1
        time.sleep(0.2) # 模拟状态更新的耗时
        return self.count, self.pid

In [14]:
# 1. 实例化 Actor (在集群中召唤一个专属的 Worker 进程)
counter_actor = StatefulCounter.remote()

# 2. 连续向【同一个】Actor 实例提交 8 个任务
refs = [counter_actor.increment_and_get.remote() for _ in range(8)]
results = ray.get(refs)

# 3. 剥离状态数据与进程 ID
states = [state for state, _ in results]
pids = {pid for _, pid in results}

print("Accumulated states:", states)
print("Unique actor pids:", pids, "count:", len(pids))

2026-04-26 11:28:21,449	INFO worker.py:2012 -- Started a local Ray instance.


Accumulated states: [1, 2, 3, 4, 5, 6, 7, 8]
Unique actor pids: {1641265} count: 1


- 在之前的 Task 示例中，8 个任务散落到 8 个不同的进程中；而在此处的 Actor 示例中，8 次调用全部被路由到了同一个 PID（1472600）。
    - 当执行 StatefulCounter.remote() 时，Ray 在底层专门 fork 了一个独占的 Worker 进程。这个进程内部维护着 self.count 这个“私有财产”，外部的任何调用都必须跨进程通信
- 从并发回到串行 (Parallelism to Serialization)
    - 由于 Actor 需要维护内部状态的一致性（避免多线程/多进程写冲突），默认情况下，发送给同一个 Actor 的所有方法调用都是串行（Sequential）执行的。
- 如果 Actor 执行的方法主要阻塞在 I/O 操作（如数据库查询、网络请求），而不是 CPU 计算，Ray 允许打破默认的串行机制。通过引入异步并发，可以实现单个 Actor 进程内的时间切片复用：

- 如果任务的核心瓶颈不在于 CPU 的高速运转，而在于漫长的 I/O 等待，Ray 允许开发者引入异步协程（Asynchronous Coroutines）。
    - 以实现 Intra-Actor Concurrency（Actor 内并发）
- 如下代码构建了一个模拟处理高延迟数据库查询的 Actor。即使是向同一个物理进程提交 10 个耗时 1 秒的任务，其总耗时依然只有 1 秒左右，而非 10 秒。

#### 并发

- 并发 (Concurrency) 绝非 并行 (Parallelism)
    - 对于 Async Actor 而言，所有的舞台都只搭建在一个单线程上。任何纯粹的 CPU 计算指令在时间轴 $t$ 上依然是严格串行的（即运算区间 $[t_1, t_2], [t_3, t_4], \dots$ 互不重叠）。
    - 并发的本质是对 I/O 空闲等待时间的巧妙交叠利用。

In [15]:
import asyncio

In [16]:
# 1. 资源边界宣告：显式配置 max_concurrency 允许并发侵入
# (注：对于 async Actor，Ray 默认放宽至 1000，但显式定义是严谨的工程实践)
@ray.remote(max_concurrency=50)
class AsyncDatabaseHandler:
    def __init__(self):
        # 内部状态：用于追踪同时在处理中的请求数量
        self.active_queries = 0
        self.pid = os.getpid()

    # 2. 语义升维：使用 async def 将方法声明为异步协程
    async def fetch_data(self, query_id):
        # 记录有几条查询正在同时并发
        self.active_queries += 1
        peak_concurrent = self.active_queries
        
        # 3. 核心机制：I/O 让权 (Yield Control)
        # await 会将当前逻辑挂起，主动把控制权交还给 Actor 内部的事件循环 (Event Loop)
        # 此时，队列中的下一个任务得以进入并执行
        await asyncio.sleep(1) # 模拟耗时 1 秒的慢查询
        
        self.active_queries -= 1
        return {
            "query_id": query_id, 
            "pid": self.pid, 
            "peak_concurrent": peak_concurrent
        }

In [17]:
# 实例化单点 Actor
db_actor = AsyncDatabaseHandler.remote()

print("Submitting 10 tasks to the single Actor...")
start_time = time.time()

# 瞬间向【同一个】 Actor 抛出 10 个查询请求
refs = [db_actor.fetch_data.remote(i) for i in range(10)]

# 阻塞收集结果（join）
results = ray.get(refs)
end_time = time.time()

print(f"Total time elapsed: {end_time - start_time:.2f} seconds\n")
for res in results[:5]:  # 仅展示前 5 个结果避免冗长
    print(res)

Submitting 10 tasks to the single Actor...
Total time elapsed: 1.14 seconds

{'query_id': 0, 'pid': 1641252, 'peak_concurrent': 9}
{'query_id': 1, 'pid': 1641252, 'peak_concurrent': 10}
{'query_id': 2, 'pid': 1641252, 'peak_concurrent': 4}
{'query_id': 3, 'pid': 1641252, 'peak_concurrent': 6}
{'query_id': 4, 'pid': 1641252, 'peak_concurrent': 7}


#### sandbox fusion 并发控制

In [18]:
import threading

```python
# sandbox_fusion_tools.py
self.execution_pool = ray.remote(ExecutionWorker)
    .options(max_concurrency=num_workers)
    .remote(enable_global_rate_limit=enable_global_rate_limit, rate_limit=rate_limit)
)
```

In [19]:
# ==========================================
# 第二层并发：服务端 (限流器 TokenBucket)
# 核心机制：并发组 (Concurrency Groups) 物理隔离
# ==========================================
@ray.remote(concurrency_groups={"acquire": 2, "release": 5})
class RateLimiter:
    def __init__(self, limit=3):
        self.limit = limit
        # 真实的线程安全信号量
        self.sema = threading.Semaphore(limit)
        
    @ray.method(concurrency_group="acquire")
    def acquire(self, task_id):
        print(f"  [Limiter] 收到任务 {task_id} 的申请...")
        # 模拟申请资源时的排队、验证或网络高延迟 (故意拉长耗时以制造拥堵)
        time.sleep(0.5) 
        self.sema.acquire()
        print(f"  [Limiter] 任务 {task_id} 成功获取令牌！")
        return True

    @ray.method(concurrency_group="release")
    def release(self, task_id):
        # 归还操作极其迅速，且拥有 5 个独立线程的 VIP 通道
        self.sema.release()
        print(f"  [Limiter] 任务 {task_id} 已归还令牌。(VIP通道放行)")
        return True

# ==========================================
# 第一层并发：客户端 (执行器 ExecutionWorker)
# 核心机制：Threaded Actor (最大并发线程池)
# ==========================================
@ray.remote(max_concurrency=10)
class ExecutionWorker:
    def __init__(self, limiter):
        self.limiter = limiter

    def execute(self, task_id):
        thread_id = threading.get_ident() % 1000  # 获取当前执行的物理线程ID简写
        print(f"[Worker-T{thread_id}] 任务 {task_id} 启动，准备强行阻塞等待令牌...")
        
        # 致命阻塞点：线程会在这里死等！
        ray.get(self.limiter.acquire.remote(task_id))
        
        # 拿到令牌后开始真正的计算 (模拟耗时)
        print(f"[Worker-T{thread_id}] 任务 {task_id} 计算中...")
        time.sleep(1) 
        
        # 计算完毕，归还令牌
        ray.get(self.limiter.release.remote(task_id))
        return f"Task {task_id} Done"

In [20]:
# 1. 实例化限流器 (系统最多只允许 3 个任务同时计算)
limiter = RateLimiter.remote(limit=3)

# 2. 实例化唯一的执行器 (内部有 10 个线程待命)
worker = ExecutionWorker.remote(limiter)

print(">>> 瞬间向单一 Worker 提交 8 个任务...")
# 3. 瞬间打满请求
refs = [worker.execute.remote(i) for i in range(8)]

# 4. 等待所有任务完成
ray.get(refs)
print(">>> 所有任务执行完毕，未发生死锁！")

>>> 瞬间向单一 Worker 提交 8 个任务...
(RateLimiter pid=1641238)   [Limiter] 收到任务 3 的申请...  [Limiter] 收到任务 2 的申请...
(RateLimiter pid=1641238) 
(ExecutionWorker pid=1641240) [Worker-T88] 任务 5 启动，准备强行阻塞等待令牌...
(ExecutionWorker pid=1641240) [Worker-T568] 任务 7 启动，准备强行阻塞等待令牌...
(ExecutionWorker pid=1641240) [Worker-T648] 任务 3 启动，准备强行阻塞等待令牌...
(ExecutionWorker pid=1641240) [Worker-T928] 任务 0 启动，准备强行阻塞等待令牌...
(ExecutionWorker pid=1641240) [Worker-T328] 任务 2 启动，准备强行阻塞等待令牌...
(ExecutionWorker pid=1641240) [Worker-T168] 任务 4 启动，准备强行阻塞等待令牌...
(ExecutionWorker pid=1641240) [Worker-T888] 任务 1 启动，准备强行阻塞等待令牌...[Worker-T408] 任务 6 启动，准备强行阻塞等待令牌...
(ExecutionWorker pid=1641240) 
(RateLimiter pid=1641238)   [Limiter] 任务 2 成功获取令牌！
(RateLimiter pid=1641238)   [Limiter] 任务 3 成功获取令牌！
(RateLimiter pid=1641238)   [Limiter] 收到任务 7 的申请...
(RateLimiter pid=1641238)   [Limiter] 收到任务 5 的申请...
(ExecutionWorker pid=1641240) [Worker-T648] 任务 3 计算中...
(ExecutionWorker pid=1641240) [Worker-T328] 任务 2 计算中...
(RateLimiter pid=16412

- 在短短几毫秒内，8 个任务同时在 ExecutionWorker 中宣告启动。注意观察，这里的进程 ID 始终是唯一的 pid=1641240，但方括号内的物理线程 ID（如 T88, T568 等）却有 8 个完全不同的数字。而且，任务启动的顺序是乱序的（5, 7, 3, 0...）。
- 服务端的“窄门”与隐形 RPC 队列
    - RateLimiter（pid=1641238）终于开始干活。
    - Worker 明明瞬间发起了 8 个申请，但 Limiter 此时却只打印了两条“收到申请”的日志
    - `@ray.remote(concurrency_groups={"acquire": 2})` 这道“窄门”在发威。由于限定了处理 acquire 的线程只有 2 个，Ray 的网关（Core Worker）在收到 8 个网络请求时，只放行了前 2 个（任务 3 和 2）。剩下的 6 个请求被悄无声息地按死在了内存的 RPC 请求队列（Request Queue）中